In [44]:
import pandas as pd
import geopandas as gpd

from util import Pipeline
from steps.legacy.adjust_targets_to_decennial import combine_targets, get_estimates_all_years

p = Pipeline('C:/Users/JKolberg/PythonProjects/control_totals/examples/legacy_luvit/configs')

In [41]:
with pd.HDFStore('C:/Users/JKolberg/PythonProjects/control_totals/examples/legacy_luvit/data/pipeline.h5') as store:
    tables = store.keys()
tables

['/adjusted_emp_change_targets',
 '/adjusted_total_pop_change_targets',
 '/adjusted_units_change_targets',
 '/block_2010_control_area_xwalk',
 '/block_control_area_xwalk',
 '/blocks',
 '/blocks_2010',
 '/control_areas',
 '/control_target_xwalk',
 '/dec_block_data',
 '/decennial_by_control_area',
 '/employment_2018_by_control_area',
 '/employment_2019_by_control_area',
 '/employment_2020_by_control_area',
 '/king_targets',
 '/kitsap_targets',
 '/ofm_block_2019',
 '/ofm_block_2019_by_control_area',
 '/ofm_parcel_control_area_xwalk',
 '/ofm_parcelized_2018',
 '/ofm_parcelized_2018_by_control_area',
 '/ofm_parcelized_2019',
 '/ofm_parcelized_2019_by_control_area',
 '/ofm_parcelized_2020',
 '/ofm_parcelized_2020_by_control_area',
 '/parcel_pts_ofm',
 '/pierce_targets',
 '/ref_projection',
 '/snohomish_targets']

In [45]:
df = p.get_table('adjusted_units_change_targets')

In [46]:
df

,target_id,start,units_chg,units_chg_adj
0,1,2019,35000,32668
1,2,2019,112000,111499
2,3,2019,12000,11342
3,4,2019,5800,5352
4,5,2019,7500,7500
5,6,2019,11260,10947
6,7,2019,3500,3000
7,8,2019,10200,9199
8,9,2019,13200,12156
9,10,2019,20000,16883


In [37]:
df.loc[df.control_id == 45]

,control_id,ofm_units,ofm_hh,ofm_gq,ofm_hhpop,ofm_total_pop
43,45,416,375,0,1039,1039


In [11]:
ofm = p.get_table('ofm_block_2019')

In [12]:
ofm['housing_units'].sum()

np.float64(1739682.1303616725)

In [23]:
ofm['housing_units'].sum()

np.float64(1739682.1303616725)

In [13]:
blk = pipeline.get_geodataframe('blocks')

In [16]:
blk = blk.merge(ofm, left_on='geoid20', right_on='block_geoid')

In [24]:
ofm['housing_units'].sum() - blk['housing_units'].sum()

np.float64(790801.3610399199)

In [29]:
len(ofm.block_geoid)

66234

In [28]:
len(ofm.block_geoid.unique())

66234

In [26]:
blk

,geoid20,geometry,block_geoid,housing_units,occupied_housing_units,group_quarters_population,household_population,index_right,control_id,control_na,county_id
0,530330003001000,POINT (1273625.064 269718.237),530330003001000,0.000,0.000,0.0,0.000,1.0,2.0,Seattle,33.0
1,530330003001001,POINT (1273498.661 270798.556),530330003001001,0.000,0.000,0.0,0.000,1.0,2.0,Seattle,33.0
2,530330003001002,POINT (1273291.416 270686.191),530330003001002,0.000,0.000,0.0,0.000,1.0,2.0,Seattle,33.0
3,530330003001003,POINT (1272762.437 270657.767),530330003001003,0.000,0.000,0.0,0.000,1.0,2.0,Seattle,33.0
4,530330003001004,POINT (1272241.692 271144.04),530330003001004,0.000,0.000,0.0,0.000,1.0,2.0,Seattle,33.0
...,...,...,...,...,...,...,...,...,...,...,...
36858,530530732003034,POINT (1265979.571 -53288.293),530530732003034,0.000,0.000,0.0,0.000,NaN,NaN,NaN,NaN
36859,530530701002050,POINT (1380702.118 -96703.428),530530701002050,0.000,0.000,0.0,0.000,149.0,303.0,Pierce Rural Ag & Resource,53.0
36860,530530730062017,POINT (1214153.554 -21366.965),530530730062017,2.997,2.994,0.0,4.333,91.0,124.0,Pierce Rural,53.0
36861,530530701002057,POINT (1383117.458 -97008.609),530530701002057,0.000,0.000,0.0,0.000,149.0,303.0,Pierce Rural Ag & Resource,53.0


In [17]:
ca = p.get_geodataframe('control_areas')

In [18]:
blk['geometry'] = blk.representative_point()

In [19]:
blk = blk.sjoin(ca, how='left')

In [20]:
blk.groupby('control_id')['housing_units'].sum()

control_id
1.0       25233.695010
2.0      168505.862001
3.0       15205.888958
4.0        7739.216048
5.0       10326.299003
             ...      
304.0      6866.535017
401.0        17.000000
402.0       783.077000
403.0      2064.707985
404.0       119.956000
Name: housing_units, Length: 154, dtype: float64

In [ ]:
pipeline.get_table('block_control_area_xwalk')

In [ ]:
pipeline.get_table('adjusted_units_change_targets')

In [ ]:


# loop through each row to calculate change from target start year to base year
for index, row in df.iterrows():
    start = int(row['start'])
    start_col = f'{target_type}_{start}'
    base_col = f'{target_type}_{base_year}'
    est_chg_col = f'est_{target_type}_chg'
    df.at[index, est_chg_col] = row[base_col] - row[start_col]

chg_adj_col = f'{target_type}_chg_adj'
chg_col = f'{target_type}_chg'
if target_type == 'emp':
    df[est_chg_col] = df[est_chg_col].fillna(0).round(0).astype(int)
    df[chg_adj_col] = (df[chg_col] - df[est_chg_col])
else:
    # fill NA, round and clip to 0 (no negative change)
    df[est_chg_col] = df[est_chg_col].fillna(0).round(0).clip(lower=0).astype(int)
    # adjust target change by subtracting est change, minimum of 0
    df[chg_adj_col] = (df[chg_col] - df[est_chg_col]).clip(lower=0)

# save adjusted targets table
table_name = f'adjusted_{target_type}_change_targets'
out_df = df[['target_id','start',chg_col,chg_adj_col]]
p.save_table(table_name,out_df)